In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import time
from pathlib import Path
from github import Github, GithubException
from datetime import datetime, timedelta, timezone

In [ ]:
import time
from requests.exceptions import ChunkedEncodingError
from urllib3.exceptions import ProtocolError

In [ ]:
TOKEN = 

OWNER = "pytorch"
REPO = "pytorch"
BRANCH = "main"

REQUEST_DELAY = 2          # seconds between API requests
SAVE_EVERY_RUNS = 50           # save progress every N commits
PER_PAGE = 100
RATE_LIMIT_BUFFER = 100

WINDOW_DAYS = 1  # 6hrs

#OUTPUT_DIR = Path("../data")
#OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMMITS_FILE = "pytorch_commits.csv"
OUTPUT_FILE = "pytorch_actions.csv"
OUTPUT_FILE_NAVI = "pytorch_navi31_actions.csv"
OUTPUT_FILE_EVENTS = "pytorch_actions_events_batch.csv"

WORKFLOW_KEYWORDS = [
    "rocm",
    "rocm-mi300",
    "inductor-rocm",
    "inductor-rocm-mi300",
    "rocm-navi31",
    "_rocm-test",
]

JOB_KEYWORDS = [
    "rocm",
    "mi300",
    "gfx",
    "navi",
]

In [ ]:
gh = Github(TOKEN, per_page=100)
repo = gh.get_repo(f"{OWNER}/{REPO}")

In [ ]:
def call_with_retry(func, *args, max_retries=5, base_sleep=30, **kwargs):
    for attempt in range(1, max_retries + 1):
        try:
            return func(*args, **kwargs)
        except (ChunkedEncodingError, ProtocolError, TimeoutError) as exc:
            if attempt == max_retries:
                raise
            sleep_seconds = base_sleep * attempt
            print(
                f"Transient connection error on attempt {attempt}/{max_retries}: {exc}. "
                f"Sleeping {sleep_seconds}s and retrying..."
            )
            time.sleep(sleep_seconds)
        except Exception:
            raise

In [ ]:
def format_github_dt(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

def iter_time_windows(start_dt, end_dt, window_days=1):
    current = start_dt

    while current < end_dt:
        next_dt = min(
            current + timedelta(days=window_days),
            end_dt,
        )

        yield current, next_dt

        current = next_dt

def wait_if_rate_limited(gh, buffer=100):
    rate = gh.get_rate_limit().core

    if rate.remaining > buffer:
        return

    reset_time = rate.reset.replace(tzinfo=timezone.utc)
    now = datetime.now(timezone.utc)
    sleep_seconds = max(
        0,
        (reset_time - now).total_seconds(),
    ) + 30

    print(
        f"Rate limit low: {rate.remaining}/{rate.limit}. "
        f"Sleeping {sleep_seconds / 60:.1f} minutes..."
    )

    time.sleep(sleep_seconds)

In [ ]:
def is_rocm_workflow(run):
    """
    Check whether workflow name is ROCm related.
    """

    fields = [
        getattr(run, "name", "") or "",
        getattr(run, "display_title", "") or "",
        getattr(run, "path", "") or "",   # if available
    ]

    text = " ".join(fields).lower()

    return any(
        keyword.lower() in text
        for keyword in WORKFLOW_KEYWORDS
    )

def is_rocm_job(job):
    fields = [
        getattr(job, "name", "") or "",
        getattr(job, "runner_name", "") or "",
        getattr(job, "runner_group_name", "") or "",
    ]

    text = " ".join(fields).lower()

    return any(
        keyword.lower() in text
        for keyword in JOB_KEYWORDS
    )

def is_relevant_rocm_record(run, job):
    """
    Keep a job if either the workflow metadata or the job metadata
    looks ROCm-related.

    This is important because some workflow run display titles are just
    commit messages like "update", while the job name may still reveal ROCm.
    """
    return is_rocm_workflow(run) or is_rocm_job(job)


In [ ]:
def load_existing_job_ids(output_file):
    if not os.path.exists(output_file):
        return set()

    old_df = pd.read_csv(output_file)

    if "job_id" not in old_df.columns:
        return set()

    return set(old_df["job_id"].dropna().astype(str))

existing_job_ids = load_existing_job_ids(OUTPUT_FILE_EVENTS)

In [ ]:
results = []

#end_dt = datetime.now(timezone.utc)
#start_dt = end_dt - timedelta(days=90)
end_dt = datetime(2026, 8, 5, 23, 59, 59, tzinfo=timezone.utc)
start_dt = datetime(2026, 7, 27, 0, 0, 0, tzinfo=timezone.utc)

print(f"Collecting workflow/job data from {start_dt} to {end_dt}")
print(f"Skipping {len(existing_job_ids)} already-saved jobs")


In [ ]:
processed_runs = 0


for window_start, window_end in iter_time_windows(
    start_dt,
    end_dt,
    WINDOW_DAYS,
):
    created_query = (
        f"{format_github_dt(window_start)}.."
        f"{format_github_dt(window_end)}"
    )

    print(f"\nWindow: {created_query}")

    try:
        #workflow_runs = repo.get_workflow_runs(
        #    created=created_query,
        #    status="completed",
        #)
        workflow_runs = call_with_retry(
            repo.get_workflow_runs,
            created=created_query,
            status="completed",
        )

        for run in workflow_runs:
            processed_runs += 1

            wait_if_rate_limited(
                gh,
                buffer=RATE_LIMIT_BUFFER,
            )

            try:
                #jobs = run.jobs()
                jobs = call_with_retry(run.jobs)

                for job in jobs:
                    job_id = str(job.id)

                    if job_id in existing_job_ids:
                        continue

                    if not is_relevant_rocm_record(run, job):
                        continue

                    results.append(
                        {
                            # Commit / run linkage
                            "commit_sha": getattr(run, "head_sha", None),
                            "commit_date": None,  # fetch later from commit API if needed

                            # Workflow information
                            "workflow_run_id": run.id,
                            "workflow_name": getattr(run, "name", None),
                            "workflow_display_title": getattr(
                                run,
                                "display_title",
                                None,
                            ),
                            "workflow_event": getattr(run, "event", None),
                            "workflow_status": getattr(run, "status", None),
                            "workflow_conclusion": getattr(
                                run,
                                "conclusion",
                                None,
                            ),
                            "workflow_run_number": getattr(
                                run,
                                "run_number",
                                None,
                            ),
                            "workflow_run_attempt": getattr(
                                run,
                                "run_attempt",
                                None,
                            ),
                            "workflow_head_sha": getattr(
                                run,
                                "head_sha",
                                None,
                            ),
                            "workflow_head_branch": getattr(
                                run,
                                "head_branch",
                                None,
                            ),
                            "workflow_created_at": getattr(
                                run,
                                "created_at",
                                None,
                            ),
                            "workflow_updated_at": getattr(
                                run,
                                "updated_at",
                                None,
                            ),

                            # Job information
                            "job_id": job.id,
                            "job_name": getattr(job, "name", None),
                            "job_status": getattr(job, "status", None),
                            "job_conclusion": getattr(
                                job,
                                "conclusion",
                                None,
                            ),
                            "started_at": getattr(job, "started_at", None),
                            "completed_at": getattr(job, "completed_at", None),
                            "runner_name": getattr(job, "runner_name", None),
                            "runner_group_name": getattr(
                                job,
                                "runner_group_name",
                                None,
                            ),

                            # URLs
                            "workflow_url": getattr(run, "html_url", None),
                            "job_url": getattr(job, "html_url", None),
                        }
                    )

                    existing_job_ids.add(job_id)

                time.sleep(REQUEST_DELAY)

            except GithubException as e:
                print(f"Error processing run {run.id}: {e}")

            if processed_runs % SAVE_EVERY_RUNS == 0:
                checkpoint_df = pd.DataFrame(results)

                if os.path.exists(OUTPUT_FILE_EVENTS):
                    old_df = pd.read_csv(OUTPUT_FILE_EVENTS)
                    checkpoint_df = pd.concat(
                        [old_df, checkpoint_df],
                        ignore_index=True,
                    ).drop_duplicates(
                        subset=["job_id"],
                        keep="last",
                    )

                checkpoint_df.to_csv(
                    OUTPUT_FILE_EVENTS,
                    index=False,
                )

                results = []

                rate = gh.get_rate_limit().core
                print(
                    f"Checkpoint saved | "
                    f"processed_runs={processed_runs} | "
                    f"API remaining={rate.remaining}/{rate.limit}"
                )

    except GithubException as e:
        print(f"Error processing window {created_query}: {e}")


# Final save
new_df = pd.DataFrame(results)

if os.path.exists(OUTPUT_FILE_EVENTS):
    old_df = pd.read_csv(OUTPUT_FILE_EVENTS)
    final_df = pd.concat(
        [old_df, new_df],
        ignore_index=True,
    ).drop_duplicates(
        subset=["job_id"],
        keep="last",
    )
else:
    final_df = new_df

final_df.to_csv(
    OUTPUT_FILE_EVENTS,
    index=False,
)

rate = gh.get_rate_limit().core

print("\nFinished.")
print(f"Saved rows: {len(final_df)}")
print(f"Saved to: {OUTPUT_FILE_EVENTS}")
print(f"API remaining: {rate.remaining}/{rate.limit}")